# R2 5′ Junction Classification

This notebook classifies 5′ junctions from ONT amplicon sequencing of R2Tg insertion sites in *Nicotiana benthamiana* 25S rDNA. Each read spanning the junction is assigned to one of three categories — anneal, join, or snap-back — following the framework of McIntyre *et al.*, 2026. The R2 5′ UTR form detected in each read (R33, full-length; R28, 5′-cleaved) is recorded alongside the junction category. Run all cells in order after completing Step 1 (`01_cutadapt_filter.sh`).

## Configuration

Set paths and thresholds here before running. `UPSTREAM_WIN` sets how many bases upstream of the R33/R28 match are extracted as the junction window for quality filtering and classification. `MIN_MEAN_Q` is the minimum mean Phred score across that window. `ALIGN_SCORE_THRESH` is the minimum local alignment score, expressed as a fraction of query length, required to classify a read as join or snap-back.

In [ ]:
import os
import re
import glob
import regex
import numpy as np
import pandas as pd
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.Align import PairwiseAligner
from collections import defaultdict

BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
FILTERED_DIR = os.path.join(BASE_DIR, 'cutadapt_filtered')
OUT_DIR     = os.path.join(BASE_DIR, 'classified')
os.makedirs(OUT_DIR, exist_ok=True)

# Reference sequences
INSERT_FASTA = os.path.join(BASE_DIR, 'insert.fasta')
RDNA_FASTA   = os.path.join(BASE_DIR, 'rDNA.fasta')

# Key sequences
PVMV        = 'ACTTCAGAGAAATTTGTAAGTTTGT'
INSERT_5END = 'GTCTAGTTACAACTGGGCAT'
ANCHOR      = 'CAAGCGCGGG'
R33         = 'TAAACGGCGGGAGTAACTATGACTCTCTTAAGG'
R28         = 'GGCGGGAGTAACTATGACTCTCTTAAGG'

UPSTREAM_WIN       = 100   # bp upstream of R33/R28 start used for classification
MIN_MEAN_Q         = 30
ALIGN_SCORE_THRESH = 0.80
MAX_R33_ERRORS     = 2     # errors allowed when finding R33/R28 in the read
MAX_INSERT_ERRORS  = 1     # errors allowed when verifying INSERT_5END follows R33/R28
MAX_R_INSERT_GAP   = 20    # max bp between end of R33/R28 and start of INSERT_5END
MAX_ANCHOR_ERRORS  = 1     # errors allowed when checking for ANCHOR upstream of R33/R28

## Reference sequences

Load the transgene insert and 25S rDNA sequences from the FASTA files provided in this repository. The insert sequence is used to detect snap-back junctions and the rDNA sequence is used to score join junctions.

In [ ]:
# Load reference sequences
insert_rec = next(SeqIO.parse(INSERT_FASTA, 'fasta'))
rdna_rec   = next(SeqIO.parse(RDNA_FASTA,   'fasta'))

INSERT_SEQ = str(insert_rec.seq).upper()
RDNA_SEQ   = str(rdna_rec.seq).upper()

print(f'Insert length : {len(INSERT_SEQ)} bp')
print(f'rDNA length   : {len(RDNA_SEQ)} bp')

## Classification functions

`find_r_in_read` searches each read for R33 or R28 followed by the transgene 5′ boundary sequence, using fuzzy matching to tolerate ONT substitution errors. R33 is searched before R28 because R28 is a suffix of R33. `fuzzy_endswith_anchor` checks whether the junction window ends with the canonical rDNA insertion site sequence, which is the criterion for anneal classification. `align_frac` scores local alignment of the junction window against a reference as a fraction of the maximum possible score.

In [ ]:
# Helper: reverse complement
def revcomp(seq: str) -> str:
    return str(Seq(seq).reverse_complement())

# Helper: local alignment score as fraction of max possible (query length)
_aligner = PairwiseAligner(mode='local', match_score=1, mismatch_score=-1,
                           open_gap_score=-2, extend_gap_score=-0.5)

def align_frac(query: str, target: str) -> float:
    if not query:
        return 0.0
    return _aligner.score(query, target) / len(query)

# Helper: exact substring search, case-insensitive
def find_seq(query: str, target: str) -> int:
    return target.upper().find(query.upper())

# ── R33/R28 detection in full read ───────────────────────────────────────────

def find_r_in_read(seq: str) -> tuple[int, int, int, str | None]:
    """Search the full read for R33 or R28 followed by INSERT_5END.

    R33 is tried before R28 because R28 is a suffix of R33; searching R33 first
    prevents R28 from matching at positions that are actually R33.

    Returns (r_start, r_end, insert_start, r_type) or (-1, -1, -1, None).
    """
    up = seq.upper()
    for r_seq, r_type in [(R33, 'R33'), (R28, 'R28')]:
        m = regex.search(f'({r_seq}){{e<={MAX_R33_ERRORS}}}', up, regex.BESTMATCH)
        if not m:
            continue
        lookahead = up[m.end() : m.end() + MAX_R_INSERT_GAP + len(INSERT_5END)]
        ins_m = regex.search(f'({INSERT_5END}){{e<={MAX_INSERT_ERRORS}}}',
                             lookahead, regex.BESTMATCH)
        if ins_m:
            insert_start = m.end() + ins_m.start()
            return m.start(), m.end(), insert_start, r_type
    return -1, -1, -1, None

# ── ANCHOR check ─────────────────────────────────────────────────────────────

def fuzzy_endswith_anchor(seq: str) -> bool:
    """True if seq ends with ANCHOR within MAX_ANCHOR_ERRORS."""
    m = regex.search(f'({ANCHOR}){{e<={MAX_ANCHOR_ERRORS}}}$', seq.upper(),
                     regex.BESTMATCH)
    return m is not None

## Per-sample classification

`process_sample` applies the full classification pipeline to a single filtered FASTQ file. Each read is first screened for plasmid-derived sequence, then searched for R33 or R28 adjacent to the transgene 5′ boundary using fuzzy matching. Reads passing a mean quality threshold across the junction window are classified as anneal (canonical rDNA anchor sequence immediately upstream of R33/R28), join (junction window aligns locally to the rDNA reference), snap-back (junction window aligns to the reverse complement of the insert), or flagged for manual inspection if no threshold is met.

In [ ]:
def process_sample(fastq_path: str) -> pd.DataFrame:
    sample = os.path.basename(fastq_path).replace('_filtered.fastq', '')
    records = list(SeqIO.parse(fastq_path, 'fastq'))
    print(f'\n=== {sample}: {len(records)} reads after cutadapt ===')

    rows = []

    for rec in records:
        seq  = str(rec.seq).upper()
        qual = rec.letter_annotations['phred_quality']
        read_id = rec.id

        # ── Plasmid filter ───────────────────────────────────────────────────
        if find_seq(PVMV, seq) != -1:
            rows.append({'read_id': read_id, 'category': 'plasmid',
                         'r_sequence': None, 'notes': ''})
            continue

        # ── Find R33/R28 + INSERT_5END in the full read ──────────────────────
        r_start, r_end, insert_start, r_sequence = find_r_in_read(seq)
        if r_sequence is None:
            rows.append({'read_id': read_id, 'category': 'discard',
                         'r_sequence': None,
                         'notes': 'R33/R28 + INSERT_5END not found'})
            continue

        # ── Quality filter: UPSTREAM_WIN bp upstream of R33/R28 start ────────
        win_start = max(0, r_start - UPSTREAM_WIN)
        win_end   = r_start
        mean_q = np.mean(qual[win_start:win_end]) if win_end > win_start else 0.0
        if mean_q < MIN_MEAN_Q:
            rows.append({'read_id': read_id, 'category': 'low_quality',
                         'r_sequence': r_sequence, 'insert_start': insert_start,
                         'notes': f'mean_Q={mean_q:.1f}'})
            continue

        # ── Extract junction region (UPSTREAM_WIN bp upstream of R33/R28) ────
        upstream = seq[win_start:win_end]

        # ── ANNEAL: ANCHOR immediately precedes R33/R28 ───────────────────────
        if fuzzy_endswith_anchor(upstream):
            rows.append({
                'read_id':      read_id,
                'category':     'anneal',
                'r_sequence':   r_sequence,
                'r_start':      r_start,
                'r_end':        r_end,
                'insert_start': insert_start,
                'upstream':     upstream,
                'notes':        ''
            })
            continue

        # ── JOIN / SNAP-BACK: align upstream to rDNA or insert ───────────────
        junc_up    = upstream.upper()
        join_score = align_frac(junc_up, RDNA_SEQ)
        snap_score = align_frac(revcomp(junc_up), INSERT_SEQ)

        if snap_score >= ALIGN_SCORE_THRESH:
            category = 'snap_back'
            notes    = f'snap_score={snap_score:.2f}'
        elif join_score >= ALIGN_SCORE_THRESH:
            category = 'join'
            notes    = f'join_score={join_score:.2f}'
        else:
            category = 'manual'
            notes    = (f'no classification: snap_score={snap_score:.2f}, '
                        f'join_score={join_score:.2f}')

        rows.append({
            'read_id':      read_id,
            'category':     category,
            'r_sequence':   r_sequence,
            'r_start':      r_start,
            'r_end':        r_end,
            'insert_start': insert_start,
            'upstream':     upstream,
            'notes':        notes
        })

    df = pd.DataFrame(rows)
    df.insert(0, 'sample', sample)
    return df

## Run classification on all samples

Process each filtered FASTQ file in `cutadapt_filtered/` and concatenate the per-sample results into a single dataframe.

In [ ]:
fastq_files = sorted(glob.glob(os.path.join(FILTERED_DIR, '*_filtered.fastq')))
print(f'Found {len(fastq_files)} filtered fastq files:')
for f in fastq_files:
    print(' ', f)

all_dfs = [process_sample(f) for f in fastq_files]
results = pd.concat(all_dfs, ignore_index=True)

## Results

Per-sample and pooled junction type counts across all four samples. Sample 4 shows a substantially higher discard rate than samples 1–3 and is excluded from the figures below, but is retained in the full results CSV.

In [ ]:
# Per-sample counts
summary = (results
           .groupby(['sample', 'category'])
           .size()
           .rename('count')
           .reset_index())

totals = results.groupby('sample').size().rename('total').reset_index()
summary = summary.merge(totals, on='sample')
summary['pct'] = (summary['count'] / summary['total'] * 100).round(1)
print(summary.to_string(index=False))

### Junction type distribution

Bar plot showing the mean percentage of each junction category across samples 1–3, with individual replicate values overlaid as points. The denominator is classified reads per sample (anneal + join + snap-back). Saved to `classified/junction_types_by_sample.pdf`.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import seaborn as sns
import matplotlib as mpl

JUNCTION_CATS = ['anneal', 'join', 'snap_back']

# --- Plot styling ---
mpl.rcParams.update({
    "font.family": "Arial",
    "font.size": 9,
    "axes.titlesize": 11,
    "axes.labelsize": 11,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
})

colors_cat = {
    'anneal':    "whitesmoke",
    'join':      "dimgrey",
    'snap_back': "black",
}

# Per-sample percentages (samples 1–3); denominator = classified reads per sample
junc = results[
    results['category'].isin(JUNCTION_CATS) & (results['sample'] != '4')
]
junc_counts = (junc
               .groupby(['sample', 'category'])
               .size()
               .unstack(fill_value=0)
               .reindex(columns=JUNCTION_CATS, fill_value=0))
junc_pct = junc_counts.div(junc_counts.sum(axis=1), axis=0) * 100
mean_pct  = junc_pct.mean()

# Long-form for swarmplot
junc_long = (junc_pct
             .reset_index()
             .melt(id_vars='sample', var_name='category', value_name='pct'))
junc_long['category'] = pd.Categorical(junc_long['category'],
                                        categories=JUNCTION_CATS, ordered=True)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(3, 5))

x = np.arange(len(JUNCTION_CATS))

# Bars at the mean
bars = ax.bar(x, mean_pct.values, width=0.75,
              color=[colors_cat[c] for c in JUNCTION_CATS],
              zorder=-2, clip_on=False, edgecolor="black")

# Swarmplot overlaid — white filled, dark outline
sns.swarmplot(data=junc_long, x='category', y='pct', order=JUNCTION_CATS,
              color='white', edgecolor='#333333', linewidth=0.8,
              size=10, zorder=-1, ax=ax)

# Mean label above each bar
for bar, pct in zip(bars, mean_pct.values):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 5,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(['Anneal', 'Join', 'Snap-back'], fontsize=10)
ax.set_xlabel('')
ax.set_ylabel('% of 5′ junctions', fontsize=10)
ax.set_ylim(0, 115)
ax.tick_params(axis='y', labelsize=9)

ax.set_axisbelow(True)
ax.tick_params(axis='y', which='both', direction='in')
plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'junction_types_by_sample.pdf'), bbox_inches='tight')
plt.show()

### Per-replicate percentages

Table of junction type percentages for each replicate and the across-replicate mean.

In [ ]:
# Table: per-replicate and mean percentages (samples 1–3)
tbl = junc_pct.T.copy()
tbl.columns = [f'sample {s} (%)' for s in tbl.columns]
tbl['mean (%)'] = junc_pct.mean().round(1)
tbl = tbl.round(1)
tbl.index = [c.replace('_', '-') for c in tbl.index]
tbl.index.name = 'junction type'
display(tbl)

### R33 vs R28 by junction type

R33 encodes the full-length R2 5′ UTR. R28 is the 5′-cleaved form produced by autocatalytic cleavage and lacks the first five nucleotides of R33. The proportion of R28-bearing reads within each junction category reflects whether 5′ UTR truncation influences junction formation. Saved to `classified/r33_r28_by_junction_type.pdf`.

In [ ]:
# R33 vs R28 breakdown within each junction type
JUNCTION_CATS = ['anneal', 'join', 'snap_back']

jdf = results[results['category'].isin(JUNCTION_CATS)].copy()

counts = (jdf
          .groupby(['category', 'r_sequence'])
          .size()
          .rename('n')
          .reset_index())

cat_totals = jdf.groupby('category').size().rename('cat_total').reset_index()
counts = counts.merge(cat_totals, on='category')
counts['pct_within_type'] = (counts['n'] / counts['cat_total'] * 100).round(1)

# Also compute each cell as % of all junction reads combined
counts['pct_of_all'] = (counts['n'] / len(jdf) * 100).round(1)

print(counts.to_string(index=False))

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pivot_pct = (counts
             .pivot(index='category', columns='r_sequence', values='pct_within_type')
             .reindex(JUNCTION_CATS)
             .reindex(columns=['R33', 'R28'])
             .fillna(0))

pivot_n = (counts
           .pivot(index='category', columns='r_sequence', values='n')
           .reindex(JUNCTION_CATS)
           .reindex(columns=['R33', 'R28'])
           .fillna(0).astype(int))

r_types = pivot_pct.columns.tolist()
cats    = pivot_pct.index.tolist()
x       = np.arange(len(cats))
width   = 0.35
colors  = {'R33': '#4C72B0', 'R28': '#DD8452'}

fig, ax = plt.subplots(figsize=(5, 4))

for i, r in enumerate(r_types):
    bars = ax.bar(x + (i - 0.5) * width, pivot_pct[r], width,
                  label=r, color=colors[r], zorder=3)
    for bar, pct, n in zip(bars, pivot_pct[r], pivot_n[r]):
        if pct > 0:
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                    f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
            ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() / 2,
                    f'n={n}', ha='center', va='center', fontsize=7,
                    color='white', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels([c.replace('_', '-') for c in cats])
ax.set_ylabel('% within junction type')
ax.set_ylim(0, 110)
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=100, decimals=0))
ax.set_title('R33 vs R28 by junction type')
ax.legend(title='5′ UTR', frameon=False)
ax.spines[['top', 'right']].set_visible(False)
ax.yaxis.grid(True, linestyle='--', linewidth=0.5, alpha=0.7, zorder=0)
ax.set_axisbelow(True)

plt.tight_layout()
fig.savefig(os.path.join(OUT_DIR, 'r33_r28_by_junction_type.pdf'), bbox_inches='tight')
plt.show()

### Manual inspection

Reads that lack the canonical anchor and do not meet the alignment threshold for join or snap-back classification are flagged for manual review. Their upstream sequences and alignment scores are saved to `classified/manual_inspection.csv`.

In [ ]:
# Reads flagged for manual inspection
manual = results[results['category'] == 'manual'][['sample', 'read_id', 'upstream', 'notes']]
print(f'\n{len(manual)} reads for manual inspection:')
if len(manual) > 0:
    display(manual)

## Save outputs

Write the full per-read classification table to `classified/junction_classifications.csv` and the manual inspection subset to `classified/manual_inspection.csv`.

In [ ]:
# Save outputs
out_csv = os.path.join(OUT_DIR, 'junction_classifications.csv')
results.to_csv(out_csv, index=False)
print(f'Full results saved to: {out_csv}')

manual_csv = os.path.join(OUT_DIR, 'manual_inspection.csv')
manual.to_csv(manual_csv, index=False)
print(f'Manual inspection reads saved to: {manual_csv}')